# Running totals of a sequence

The inclusive prefix sum of a sequence has, at each position, the total of every element up to
and including that position. NumPy provides it as `cumsum`. Each total depends on the one before
it, which makes the calculation hard to share between workers.

In [ ]:
import numpy as np


def scan(x):
    """The inclusive prefix sum: out[i] = x[0] + x[1] + ... + x[i]."""
    return np.cumsum(x)

In [ ]:
print(scan(np.array([3.0, 1.0, 4.0, 1.0, 5.0])))

The declaration below tells the harness how to call the function and how to check a routine
against it. `make_inputs` draws random numbers between -0.5 and 0.5 and adds to all of them one
random drift between -0.1 and 0.1, so that the totals rise or fall by a different amount each
time. A few elements are set to exactly -0.5, 0 or 0.5. A routine that shares the work between
workers adds the numbers in a different order. The tolerance is 5e-12 of the largest total,
which is about twelve times the largest difference, 4e-13, between the function and a routine
that uses eight workers on 50 million numbers.

In [ ]:
def make_inputs(size, rng):
    x = rng.uniform(-0.5, 0.5, size) + rng.uniform(-0.1, 0.1)
    special = rng.random(size) < 0.05
    x[special] = rng.choice([-0.5, 0.0, 0.5], special.sum())
    return {"x": x}


ROUTINE = {
    "function": scan,
    "make_inputs": make_inputs,
    "arguments": {"x": "float64[n]"},
    "results": {"out": "float64[n]"},
    "test_sizes": [1, 1_000, 1_000_000],
    "benchmark_size": 50_000_000,
    "tolerance": 5e-12,
}